# Experiment 1 · Notebook 03: Group-aware split and trivial baselines

*J26-IT-304 PP1. Baseline confusion classifier (TF-IDF + Logistic Regression). Sections 6–7.*

The last stage before any model is trained. It proves the train/test split shares no question, scores the trivial baselines the model must beat, and freezes the OOD probe set before any model exists.

Section numbers run continuously across the Experiment 1 series, so a reference such as "EDA §3.5" or "section 8" points to the same place in every notebook:

| Notebook | Sections | Stage |
|---|---|---|
| `01_data_and_eda` | §1–§3 | provenance, load, exploratory analysis, leakage demonstration |
| `02_cleaning_and_labels` | §4–§5 | deduplication, group key, label mapping, proxy caveat |
| `03_split_and_baselines` | §6–§7 | group-aware split, trivial baselines, frozen OOD probe |
| `04_model_training_and_evaluation` | §8–§9 | TF-IDF + LR, group-aware CV, the single test evaluation |

Each notebook runs on its own from a clean kernel (kernel `exp1-confusion`). Its first code cell rebuilds its inputs with the same `confusion_baseline` functions the previous stage used, so the stages cannot drift apart. Run them in order 01 → 04 to regenerate every file in `reports/experiments/`.

**Setup.** Imports, and the input of this stage: the primary-mapped frame from notebook 02 §5, rebuilt with the same package functions and checked against its known size.

In [1]:
import numpy as np
import pandas as pd

from confusion_baseline import REPORTS_DIR, SEED
from confusion_baseline.data import load_raw
from confusion_baseline.prep import clean, map_labels, save_processed
from confusion_baseline.text import normalize_key

pd.set_option("display.width", 200)
FIG_DIR = REPORTS_DIR / "experiments"

# Input of this stage: the primary-mapped frame of notebook 02 (section 5), rebuilt with the same functions.
df, followup_quarantine, _ = load_raw()
del followup_quarantine
clean_df, _ = clean(df)
y_primary = map_labels(clean_df, "primary")
assert len(y_primary) == 1076 and y_primary["group_key"].nunique() == 360
print("input frame:", y_primary.shape, "| groups:", y_primary["group_key"].nunique(),
      f"| p(confused) = {y_primary['y'].mean():.4f}")

input frame: (1076, 7) | groups: 360 | p(confused) = 0.6097


## 6. Group-aware split

**Why this matters.** Each question has about 3 answers (1 to 12). A random split puts answers to the
*same question* on both sides: EDA section 3.2 measured that 93.5% of validation rows would have their
question in training. The model could then recognise the question's wording instead of judging the
answer, and the F1 would measure memorisation. The examiner question this answers is: **"How do you know
this isn't memorising the questions?"** The answer is the printed, passing assertion below, not a sentence.

The split is sklearn `GroupShuffleSplit` (n_splits=1, test_size=0.2, random_state=42) with
`groups = group_key`. It is **not stratified**, because the spec names `GroupShuffleSplit`, and the artifact
metadata will record it. `test_size` is a fraction of **groups**, so the test *row* share will not be exactly
20%. The seed is fixed once. If the resulting class balance looks unlucky, the split is **not** redrawn
with another seed: reseeding until a split looks good is a forbidden practice in the spec.

In [2]:
from confusion_baseline.split import group_split, split_fingerprint
from confusion_baseline.evaluate import metrics

train, test = group_split(y_primary, test_size=0.2, seed=SEED)

shared = set(train.group_key) & set(test.group_key)
assert not shared, shared
print(f"PASS: train and test share 0 questions ({train.group_key.nunique()} train groups, "
      f"{test.group_key.nunique()} test groups, intersection = {len(shared)})")

assert set(train.row_id).isdisjoint(test.row_id)
assert set(train.row_id) | set(test.row_id) == set(y_primary.row_id)
assert len(train) + len(test) == len(y_primary) == 1076
print(f"PASS: row_ids are disjoint and train + test = all {len(y_primary)} row_ids ({len(train)} + {len(test)})")

PASS: train and test share 0 questions (288 train groups, 72 test groups, intersection = 0)
PASS: row_ids are disjoint and train + test = all 1076 row_ids (860 + 216)


**Why.** The group assertion covers exact questions. Two other leak routes are checked: the *same
answer text* on both sides, and the two near-duplicate answer pairs the EDA found across different
questions (section 3.3). A split by question cannot separate those, so any that cross are reported as
known residual leakage, and no rows are moved.

In [3]:
ans_train = set(train["Answer"].map(normalize_key))
ans_test = set(test["Answer"].map(normalize_key))
print("normalised Answer texts present on BOTH sides:", len(ans_train & ans_test), "(expected 0)")

side = {r: "train" for r in train.row_id} | {r: "test" for r in test.row_id}
for a, b in [(1028, 1033), (920, 932)]:
    verdict = "same side" if side[a] == side[b] else "CROSSES the split"
    print(f"near-duplicate pair {a}/{b}: row {a} -> {side[a]}, row {b} -> {side[b]}  => {verdict}")

normalised Answer texts present on BOTH sides: 0 (expected 0)
near-duplicate pair 1028/1033: row 1028 -> train, row 1033 -> test  => CROSSES the split
near-duplicate pair 920/932: row 920 -> train, row 932 -> train  => same side


**Why.** The sizes table shows how the group split came out: rows, groups, row share and confused
proportion per side. The test confused proportion is compared with the overall one, because a group split
without stratification can drift.

In [4]:
def size_row(name, d):
    return {"split": name, "rows": len(d), "groups": d["group_key"].nunique(),
            "row share": round(len(d) / len(y_primary), 4), "confused proportion": round(float(d["y"].mean()), 4)}

sizes = pd.DataFrame([size_row("train", train), size_row("test", test), size_row("all", y_primary)])
display(sizes)
gap = float(test["y"].mean() - y_primary["y"].mean())
print(f"test confused proportion minus overall = {gap:+.4f}  (flag threshold 0.05: {'EXCEEDED' if abs(gap) > 0.05 else 'not exceeded'})")
print("test confused count / rows:", int(test['y'].sum()), "/", len(test))

fp = split_fingerprint(test)
print("split fingerprint (sha256 of sorted test group keys):", fp)

out = pd.concat([train.assign(split="train"), test.assign(split="test")]).sort_values("row_id").reset_index(drop=True)
path = save_processed(out, "split_primary")
print("saved:", path.relative_to(path.parents[2]).as_posix(), "| columns:", list(out.columns))

,split,rows,groups,row share,confused proportion
0,train,860,288,0.7993,0.6128
1,test,216,72,0.2007,0.5972
2,all,1076,360,1.0000,0.6097


test confused proportion minus overall = -0.0124  (flag threshold 0.05: not exceeded)
test confused count / rows: 129 / 216
split fingerprint (sha256 of sorted test group keys): 511fadfecbff1c84dad9f22c4ecdd51052b13c6c1187c6f60d2468342905e44c
saved: data/processed/split_primary.parquet | columns: ['row_id', 'Question', 'Answer', 'Correctness', 'group_key', 'label', 'y', 'split']


**Reading section 6.** The assertion passes: 288 train groups and 72 test groups share no question (860 + 216
rows = all 1,076). So the test score cannot come from having seen the test questions, which is the
answer to "how do you know this isn't memorising the questions?". No normalised answer text appears on both
sides.

**Known residual leakage.** The near-duplicate pair **1028 / 1033** (two `floor()` questions with almost
identical answers, cosine 0.931) **crosses the split**: row 1028 is in train and row 1033 is in test. This is the
question-family effect noted in section 3.3. Both rows are Correct, so the effect is at most 1 test row out of 216. It
is documented and rows are not moved. The other pair (920 / 932) stays on one side (both train).

**Class balance.** The test confused proportion is 0.5972 against 0.6097 overall (difference -0.0124), well
inside the 0.05 flag threshold, so there is nothing to caveat. The test row share is 20.07% (216 of 1,076), close
to 20% because the groups are small. The split is **not** re-drawn under any circumstances; this one seed (42) is the
test set, and the fingerprint above lets a later unit prove it evaluates on the same one.

## 7. Trivial baselines

**Why.** A raw F1 is uninterpretable without knowing what trivial strategies score. With 61% confused,
"always confused" already reaches F1(confused) of about 0.76. The model's claim will be its **margin**
over these rows, not its absolute number. Both baselines are `sklearn.dummy.DummyClassifier`: they ignore
the text, are fitted on **train** and evaluated on **test**. No TF-IDF and no model are used yet.
`zero_division=0` is used for precision, so a constant "understood" predictor (which never predicts
confused) gets precision 0 instead of an undefined value.

In [5]:
from sklearn.dummy import DummyClassifier

X_tr, X_te = np.zeros((len(train), 1)), np.zeros((len(test), 1))   # Dummy ignores features
systems = {
    "Majority class": DummyClassifier(strategy="most_frequent"),
    "Stratified random (seed 42)": DummyClassifier(strategy="stratified", random_state=SEED),
}
rows = []
for name, clf in systems.items():
    clf.fit(X_tr, train["y"])
    m = metrics(test["y"], clf.predict(X_te))
    rows.append({"System": name, "Accuracy": m["accuracy"], "Precision": m["precision_confused"],
                 "Recall": m["recall_confused"], "F1 (confused)": m["f1_confused"], "Macro F1": m["macro_f1"]})
baselines = pd.DataFrame(rows).round(4)
display(baselines)
print("majority class predicted:", systems["Majority class"].classes_[np.argmax(systems["Majority class"].class_prior_)], "(1 = confused)")

csv_path = FIG_DIR / "baselines_primary.csv"
baselines.to_csv(csv_path, index=False)
print("saved:", csv_path.relative_to(FIG_DIR.parent.parent).as_posix())

,System,Accuracy,Precision,Recall,F1 (confused),Macro F1
0,Majority class,0.5972,0.5972,1.0000,0.7478,0.3739
1,Stratified random (seed 42),0.4583,0.5441,0.5736,0.5585,0.4289


majority class predicted: 1 (1 = confused)
saved: reports/experiments/baselines_primary.csv


**Why.** One random draw of the stratified baseline on about 200 test rows is noisy. The table row stays
at seed 42; the note below shows how much that row could move, so a later comparison is not misread.

In [6]:
f1s = []
for s in range(1000):
    c = DummyClassifier(strategy="stratified", random_state=s).fit(X_tr, train["y"])
    f1s.append(metrics(test["y"], c.predict(X_te))["f1_confused"])
f1s = np.array(f1s)
print(f"stratified baseline F1(confused) over seeds 0-999: mean = {f1s.mean():.4f} +/- {f1s.std(ddof=1):.4f} (sd), "
      f"min {f1s.min():.4f}, max {f1s.max():.4f}")
print(f"table row (seed {SEED}) = {baselines.loc[1, 'F1 (confused)']:.4f}")
p_test = float(test["y"].mean())
print(f"constant 'always confused' F1(confused) on THIS test set = 2p/(1+p) = {2 * p_test / (1 + p_test):.4f}  (p = {p_test:.4f})")

stratified baseline F1(confused) over seeds 0-999: mean = 0.6037 +/- 0.0318 (sd), min 0.5134, max 0.6985
table row (seed 42) = 0.5585
constant 'always confused' F1(confused) on THIS test set = 2p/(1+p) = 0.7478  (p = 0.5972)


**Reading section 7.** On this test set (216 rows, 59.7% confused) **"always confused" scores F1(confused) =
0.7478** (accuracy 0.5972, macro F1 0.3739). It looks strong on the confused class while never recognising
an understood answer, which is why macro F1 is reported next to it. Stratified random guessing scores
0.5585 at seed 42, but over seeds 0-999 it averages 0.6037 +/- 0.0318 (range 0.5134 to 0.6985), so the seed-42 row is a
below-average draw and **must not be used as the bar**. The real floor for the model is the constant predictor,
**F1(confused) = 0.7478 and macro F1 = 0.3739** on this test set. Beating 0.75 on F1(confused) is only
meaningful together with macro F1 and the margin over this row.

### The out-of-distribution probe set

**Why.** `probe/ood_probe.yaml` holds 40 hand-written utterances of the kind the *deployed* tutor will
receive and the training data contains none of (no hedging, every answer a confident statement). It was
written in this unit, before any model exists, and without looking at the training answers, so it cannot
have been tailored to what a model gets right or wrong. This cell only loads it and counts it.

**Frozen at U4; evaluated in section 11; never used for training.** No classifier is run on it here.

In [7]:
from confusion_baseline.probe import load_probe, probe_sha256

probe = load_probe()
print("probe items:", len(probe))
display(pd.crosstab(probe["category"], probe["expected"], margins=True))
print("probe file sha256:", probe_sha256())

probe items: 40


expected,confused,understood,All
category,,,
explicit_confusion,10,0,10
explicit_understanding,0,10,10
hedged_attempt,10,0,10
topic_confusion,10,0,10
All,30,10,40


probe file sha256: cbccf0a0c0d10eded7ac31df41d4af1afde49c66242f685a6cfea41e1e9908f5


**Probe composition.** The probe has 30 expected-confused items (three categories of 10) and 10
expected-understood items, because the brief asks for 10 per category. It is therefore 3 : 1 towards
confused, so an "always confused" predictor would score accuracy 0.75 on it. When the probe is evaluated in
section 11 it should be reported **per category** (recall on each confused category, specificity on
`explicit_understanding`), not as one accuracy.

## Findings of notebook 03

| Topic | Finding | Where |
|---|---|---|
| Split | `GroupShuffleSplit` by question, seed 42: train 860 rows / 288 questions, test 216 rows / 72 questions. **Printed assertion: 0 shared questions.** | §6 |
| Residual overlap | 0 answer texts on both sides. One near-duplicate pair (1028 train / 1033 test, two `floor()` questions, both Correct) crosses the split: at most 1 of 216 test rows. Documented, not moved. | §6 |
| Balance | Test is 59.7% confused vs 61.0% overall (−0.012). Not re-split. Test fingerprint `511fadfe…` | §6 |
| Baselines | Majority class: accuracy 0.5972, F1(confused) **0.7478**, macro F1 **0.3739**. Stratified random (seed 42): F1(confused) 0.5585; over 1,000 seeds 0.6037 ± 0.0318. | §7 |
| The bar | Any model must be judged by its **margin** over F1(confused) 0.7478 **and** macro F1 0.3739. | §7 |
| OOD probe | 40 hand-written utterances (30 confused, 10 understood), frozen here **before any model existed** (sha256 `cbccf0a0…`). Never trained on. | §7 |